In [42]:
import os
os.environ['JAVA_HOME'] = "/usr/lib/jvm/java-21-openjdk-amd64"

In [43]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import RegexTokenizer, HashingTF, IDF
from pyspark.sql.functions import col, udf, lit, regexp_extract, coalesce, when
from pyspark.sql.types import FloatType, IntegerType
import numpy as np
spark = SparkSession.builder.master('local[*]').getOrCreate()

spark.conf.set('spark.sql.repl.eagerEval.enabled', True)

spark

In [ ]:
# load dataset 
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .load("dataset/glints_dataset.csv")

df.printSchema()
df.show(5)

root
 |-- job_title: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- education_req: string (nullable = true)
 |-- salary_range: string (nullable = true)
 |-- job_requirements: string (nullable = true)
 |-- job_responsibilities: string (nullable = true)
 |-- posted_date: timestamp (nullable = true)
 |-- source_platform: string (nullable = true)
 |-- category_scraped: string (nullable = true)

+--------------------+--------------------+----------------+---------+----------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------+--------------------+
|           job_title|        company_name|        location| job_type|experience_level|  education_req|        salary_range|    job_requirements|job_responsibilities|         posted_date|source_platform|    category_scraped|
+

In [45]:
df.printSchema()

root
 |-- job_title: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- education_req: string (nullable = true)
 |-- salary_range: string (nullable = true)
 |-- job_requirements: string (nullable = true)
 |-- job_responsibilities: string (nullable = true)
 |-- posted_date: timestamp (nullable = true)
 |-- source_platform: string (nullable = true)
 |-- category_scraped: string (nullable = true)



In [46]:
from pyspark.sql.functions import col, sum

# Get the count of nulls in each column
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+---------+------------+--------+--------+----------------+-------------+------------+----------------+--------------------+-----------+---------------+----------------+
|job_title|company_name|location|job_type|experience_level|education_req|salary_range|job_requirements|job_responsibilities|posted_date|source_platform|category_scraped|
+---------+------------+--------+--------+----------------+-------------+------------+----------------+--------------------+-----------+---------------+----------------+
|        0|           0|       0|       0|               0|            0|           0|               0|                   0|          0|              0|               0|
+---------+------------+--------+--------+----------------+-------------+------------+----------------+--------------------+-----------+---------------+----------------+



In [47]:
df.select('job_type').distinct().show()

+-------------+
|     job_type|
+-------------+
|    PART_TIME|
|    FULL_TIME|
|   INTERNSHIP|
|     CONTRACT|
|PROJECT_BASED|
+-------------+



In [48]:
# This shows you the unique job titles in your dataset
df.select("education_req").distinct().show(truncate=False)

+----------------------+
|education_req         |
+----------------------+
|SECONDARY_SCHOOL      |
|PRIMARY_SCHOOL        |
|MASTER_DEGREE         |
|DIPLOMA               |
|PROFESSIONAL_EDUCATION|
|DOCTORATE             |
|BACHELOR_DEGREE       |
|HIGH_SCHOOL           |
+----------------------+



In [49]:
df.select("location").distinct().show(truncate=False)

+----------------+
|location        |
+----------------+
|Tebet           |
|Purwodadi       |
|Menteng         |
|Batu Raja Timur |
|Rajeg           |
|Cicalengka      |
|Ulujadi         |
|Pangkalan Baru  |
|Panarukan       |
|Balige          |
|Mundu           |
|Pasawahan       |
|Tangerang       |
|Kemantren Kraton|
|Padang Barat    |
|Mejayan         |
|Bukitintan      |
|Bojong          |
|Tempe           |
|Gatak           |
+----------------+
only showing top 20 rows


In [50]:
# register dataframe as a temporary sql table
df.createOrReplaceGlobalTempView('my_table')

# Preprocessing

In [51]:
# drop data duplikat dan data dimana scrapper gagal untuk mengambil kolom job_responsibilities (karena kendala jaringan).
def clean_df(df):
    df_clean = df.dropDuplicates().filter(~col("job_responsibilities").startswith("Error:"))

    # drop semua data yang memiliki null values
    df_clean = df_clean.dropna()

    return df_clean

df_clean = clean_df(df)

print("Jumlah baris data sebelum di cleaning:", df.count())
print("Jumlah baris data setelah di cleaning:", df_clean.count())

Jumlah baris data sebelum di cleaning: 75748


Jumlah baris data setelah di cleaning: 70486


In [52]:
# buat kolom baru: min_experience_level, max_experience_level, min_salary, dan max_salary
def create_cols(df):
    raw_min_exp = regexp_extract(col("experience_level"), r"(\d+)", 1)
    raw_max_exp = regexp_extract(col("experience_level"), r"-(\d+)", 1)
    
    raw_min_sal = regexp_extract(col("salary_range"), r"(\d+)", 1)
    raw_max_sal = regexp_extract(col("salary_range"), r"-\s*(\d+)", 1)

    df_new = df.withColumn(
        "min_experience_level",
        when(raw_min_exp == "", None).otherwise(raw_min_exp).cast("int")
    ).withColumn(
        "max_experience_level",
        coalesce(
            when(raw_max_exp == "", None).otherwise(raw_max_exp).cast("int"),
            col("min_experience_level")
        )
    ).withColumn(
        "min_salary",
        when(raw_min_sal == "", None).otherwise(raw_min_sal).cast("long")
    ).withColumn(
        "max_salary",
        coalesce(
            when(raw_max_sal == "", None).otherwise(raw_max_sal).cast("long"),
            col("min_salary")
        )
    ).drop("salary_range", "experience_level")

    return df_new

df_new = create_cols(df_clean)

In [53]:
def encode_edu_req(education):
    education_map = {
        "PRIMARY_SCHOOL": 1,
        "SECONDARY_SCHOOL": 2,
        "HIGH_SCHOOL": 3,
        "DIPLOMA": 4,
        "BACHELOR_DEGREE": 5,
        "PROFESSIONAL_EDUCATION": 6,
        "MASTER_DEGREE": 7,
        "DOCTORATE": 8
    }

    if education is None:
        return 0
    return education_map.get(education, 0)

# register fungsi sebagai User Defined Function dengan output integer
encode_edu_udf = udf(encode_edu_req, IntegerType())

df_encoded = df_new.withColumn("education_req", encode_edu_udf(col("education_req")))

df_encoded.show()

+--------------------+--------------------+---------------+---------+-------------+--------------------+--------------------+--------------------+---------------+--------------------+--------------------+--------------------+----------+----------+
|           job_title|        company_name|       location| job_type|education_req|    job_requirements|job_responsibilities|         posted_date|source_platform|    category_scraped|min_experience_level|max_experience_level|min_salary|max_salary|
+--------------------+--------------------+---------------+---------+-------------+--------------------+--------------------+--------------------+---------------+--------------------+--------------------+--------------------+----------+----------+
|HELPER DRIVER / W...|PT Shield On Serv...|         Gambir|FULL_TIME|            3|Stock Opname, Log...|Deskripsi pekerja...|2026-03-10 11:28:...|         Glints|Supply Chain, Log...|                   3|                   5|   4000000|   4000000|
|Helper 

In [54]:
from pyspark.sql.functions import col, sum, when, trim

# Build a list of counting expressions for every column
blank_counts = [
    sum(
        when((trim(col(c)) == "") | (col(c).isNull()), 1).otherwise(0)
    ).alias(c) 
    for c in df.columns
]

# Run the aggregation
df_clean.select(blank_counts).show()

+---------+------------+--------+--------+----------------+-------------+------------+----------------+--------------------+-----------+---------------+----------------+
|job_title|company_name|location|job_type|experience_level|education_req|salary_range|job_requirements|job_responsibilities|posted_date|source_platform|category_scraped|
+---------+------------+--------+--------+----------------+-------------+------------+----------------+--------------------+-----------+---------------+----------------+
|        0|           0|       0|       0|               0|            0|           0|               0|                   0|          0|              0|               0|
+---------+------------+--------+--------+----------------+-------------+------------+----------------+--------------------+-----------+---------------+----------------+



In [55]:
# user input
user_skills = "Microsoft Excel, Data Entry, Administration, Problem Solving"
user_years_of_experience = 1
user_highest_education = "HIGH_SCHOOL"

def df_user(df, job_type=None, highest_education=None, years_of_experience=None, salary_target=None, job_category=None):
    df_filtered = df
    if job_type is not None:
        df_filtered.filter(col("job_type") == job_type)
    
    if highest_education is not None:
        df_filtered.filter(col("education_req") <= highest_education)
    
    if years_of_experience is not None:
        df_filtered.filter(
            (col("min_experience_level") <= years_of_experience) &
            (col("max_experience_level") >= years_of_experience)
        )

    if salary_target is not None:
        df_filtered = df_filtered.filter(
            (col("min_salary") <= salary_target) & 
            (col("max_salary") >= salary_target)
        )

    if job_category is not None:
        df_filtered = df_filtered.filter(col("job_category") == job_category)
        
    return df_filtered

df_filtered = df_user(
    df_encoded,
    job_type=None,
    highest_education=user_highest_education,
    years_of_experience=user_years_of_experience,
    salary_target=None,
    job_category=None
)

In [56]:
tokenizer = RegexTokenizer(inputCol="job_requirements", outputCol="skills_tokens", pattern=",\s*")

df_tokenized = tokenizer.transform(df_filtered)

In [57]:
hashingTF = HashingTF(inputCol="skills_tokens", outputCol="rawFeatures", numFeatures=1000)
featurizedData = hashingTF.transform(df_tokenized)

idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)

rescaledData = idfModel.transform(featurizedData)

In [58]:
user_df = spark.createDataFrame([(user_skills,)], ["job_requirements"])

user_tokenized = tokenizer.transform(user_df)
user_featurized = hashingTF.transform(user_tokenized)
user_rescaled = idfModel.transform(user_featurized)

user_vector = user_rescaled.select("features").head()[0]

In [59]:
def cosine_similarity(v1):
    dot_product = float(v1.dot(user_vector))
    norm_v1 = float(v1.norm(2))
    norm_user = float(user_vector.norm(2))
    
    if norm_v1 == 0 or norm_user == 0:
        return 0.0
        
    return dot_product / (norm_v1 * norm_user)

cosine_similarity_udf = udf(cosine_similarity, FloatType())

In [62]:
df_with_similarity = rescaledData.withColumn("similarity_score", cosine_similarity_udf(col("features")))

df_sorted = df_with_similarity.orderBy(col("similarity_score").desc())

top_recommendations = df_sorted.select("job_title", "company_name", "job_requirements", "category_scraped", "job_type", "min_salary", "max_salary", "min_experience_level", "similarity_score").limit(10)

top_recommendations.show(truncate=False)

# spark.stop()

+-------------------+-------------------------------+-------------------------------------------+--------------------+---------+----------+----------+--------------------+----------------+
|job_title          |company_name                   |job_requirements                           |category_scraped    |job_type |min_salary|max_salary|min_experience_level|similarity_score|
+-------------------+-------------------------------+-------------------------------------------+--------------------+---------+----------+----------+--------------------+----------------+
|Admin Warehouse    |PT. Pillar Brite Care          |Data Entry, Administration, Microsoft Excel|Administrative & HR |CONTRACT |2800000   |3200000   |1                   |0.8652589       |
|Admin Sales Group  |PT Indocitra Pacific (Pestcare)|Data Entry, Microsoft Excel, Administration|Administrative Staff|FULL_TIME|5300000   |5700000   |1                   |0.8652589       |
|Admin Operasional  |Antartika - Unit Ice Bali      |Mi

In [61]:
df_filtered.filter(col("job_title") == "Customer Service (E-Commerce)")

job_title,company_name,location,job_type,education_req,job_requirements,job_responsibilities,posted_date,source_platform,category_scraped,min_experience_level,max_experience_level,min_salary,max_salary
